<a href="https://colab.research.google.com/github/lgomezan-dev/practica-ssh-git/blob/main/rocketalfa_1_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install cadquery ipywidgets

  Using cached cadquery-2.8.0-py3-none-any.whl.metadata (14 kB)
  Using cached cadquery_ocp-7.9.3.1.1-cp313-cp313-manylinux_2_31_x86_64.whl.metadata (831 bytes)
  Using cached trame-4.0.0-py3-none-any.whl.metadata (8.7 kB)
Using cached cadquery-2.8.0-py3-none-any.whl (201 kB)
Using cached cadquery_ocp-7.9.3.1.1-cp313-cp313-manylinux_2_31_x86_64.whl (67.8 MB)
Using cached trame-4.0.0-py3-none-any.whl (27 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 22.6 MB/s eta 0:00:00


In [2]:
import gzip
import os
import math
import xml.etree.ElementTree as ET
import zipfile
from io import BytesIO
import re
import google.colab.files as colab_files
from IPython.display import clear_output, display
import ipywidgets as widgets

try:
    import cadquery as cq
    CADQUERY_AVAILABLE = True
except ImportError:
    CADQUERY_AVAILABLE = False


def remove_duplicate_points(pts, tol=1e-3):
    """Filtra puntos 2D duplicados (distancia < tol) y garantiza X > 0 para evitar fallos de revolución CAD."""
    clean_pts = []
    for pt in pts:
        x_val = max(0.001, float(pt[0]))
        y_val = float(pt[1])
        new_pt = (x_val, y_val)
        if not clean_pts:
            clean_pts.append(new_pt)
        else:
            last_pt = clean_pts[-1]
            dist = math.hypot(new_pt[0] - last_pt[0], new_pt[1] - last_pt[1])
            if dist > tol:
                clean_pts.append(new_pt)
    return clean_pts


def extract_xml_content(file_bytes):
    """Descomprime y extrae el archivo XML embebido en contenedores .ork (GZIP, ZIP o texto plano)."""
    try:
        return gzip.decompress(file_bytes)
    except Exception:
        pass
    try:
        with zipfile.ZipFile(BytesIO(file_bytes)) as zip_file:
            for internal_filename in zip_file.namelist():
                if internal_filename.endswith('.xml') or internal_filename == 'rocket.xml':
                    return zip_file.read(internal_filename)
            if zip_file.namelist():
                return zip_file.read(zip_file.namelist()[0])
    except Exception:
        pass
    return file_bytes


def read_xml_float(element, tag_name, default_val=0.0):
    """Lee numéricos flotantes de un nodo XML filtrando prefijos de texto o notación científica."""
    found = element.find(tag_name)
    if found is not None and found.text:
        text = found.text.strip()
        match = re.search(r'-?\d+\.?\d*(?:[eE][-+]?\d+)?', text)
        if match:
            try:
                return float(match.group())
            except ValueError:
                return default_val
    return default_val


def read_xml_angle_degrees(item, default_val=0.0):
    """Extrae la orientación angular radial en grados sexagesimales desde nodos de OpenRocket."""
    for tag in ('angleoffset', 'radialdirection', 'rotation'):
        found = item.find(tag)
        if found is not None and found.text:
            match = re.search(r'-?\d+\.?\d*(?:[eE][-+]?\d+)?', found.text.strip())
            if match:
                try:
                    return float(match.group())
                except ValueError:
                    continue
    return default_val


def get_component_z_offset(item, parent_length):
    """Calcula la posición Z absoluta de un subcomponente según su referencia ('top', 'middle', 'bottom')."""
    pos_elem = item.find('position')
    if pos_elem is not None and pos_elem.text:
        try:
            pos_val = float(pos_elem.text) * 1000.0
        except ValueError:
            pos_val = 0.0
        pos_type = pos_elem.attrib.get('type', 'top').lower()
    else:
        pos_val = 0.0
        pos_type = 'top'

    if pos_type == 'bottom':
        return parent_length + pos_val
    elif pos_type == 'middle':
        return (parent_length / 2.0) + pos_val
    else:
        return pos_val


def parse_freeform_fin_points(item_element):
    """Parsea las coordenadas 2D (r, z) para polígonos de aletas de forma libre (Freeform)."""
    pts = []
    point_nodes = item_element.findall('.//point')
    for p in point_nodes:
        try:
            x = float(p.attrib.get('x', 0.0)) * 1000.0
        except (TypeError, ValueError):
            x = 0.0
        try:
            y = float(p.attrib.get('y', 0.0)) * 1000.0
        except (TypeError, ValueError):
            y = 0.0
        pts.append((y, x))
    return pts


def get_profile_points(shape_type, shape_param, r, l, steps=28):
    """Genera la perfilería aerodinámica 2D (conica, elíptica, parabólica, Haack u ojival) para revoluciones 3D."""
    shape_type = (shape_type or 'ogive').lower()
    pts = [(0.001, 0.0)]

    if shape_type in ('conical', 'cone'):
        for i in range(1, steps + 1):
            x = (l / steps) * i
            pts.append((max(0.001, r * x / l), x))

    elif shape_type in ('elliptical', 'ellipse'):
        for i in range(1, steps + 1):
            x = (l / steps) * i
            ri = r * math.sqrt(max(0.0, 1 - ((x - l) / l) ** 2))
            pts.append((max(0.001, ri), x))

    elif shape_type in ('parabolic', 'power', 'powerseries'):
        n = shape_param if (shape_param is not None and shape_param > 0) else 0.5
        n = max(0.05, min(1.0, n))
        for i in range(1, steps + 1):
            x = (l / steps) * i
            ri = r * ((x / l) ** n)
            pts.append((max(0.001, ri), x))

    elif shape_type in ('haack', 'vonkarman', 'karman'):
        K = shape_param if shape_param is not None else (1 / 3.0 if 'karman' in shape_type else 0.0)
        for i in range(1, steps + 1):
            x = (l / steps) * i
            theta = math.acos(max(-1.0, min(1.0, 1 - 2 * x / l)))
            ri = (r / math.sqrt(math.pi)) * math.sqrt(
                max(0.0, theta - math.sin(2 * theta) / 2.0 + K * (math.sin(theta) ** 3)))
            pts.append((max(0.001, ri), x))

    else:
        rho_tangent = (r ** 2 + l ** 2) / (2 * r)
        if shape_param is not None and 0 < shape_param < 1:
            rho = rho_tangent / max(0.01, (1 - shape_param))
        else:
            rho = rho_tangent
        for i in range(1, steps + 1):
            x = (l / steps) * i
            ri = math.sqrt(max(0.0, rho ** 2 - (l - x) ** 2)) + r - rho
            pts.append((max(0.001, ri), x))

    pts.append((0.001, l))
    return remove_duplicate_points(pts)


def parse_ork_geometry(root_node, current_z=0.0, component_list=None, parent_radius=0.0, parent_length=0.0):
    """Parsea la jerarquía XML de OpenRocket y construye la lista de componentes geométricos."""
    if component_list is None:
        component_list = []

    for item in root_node:
        tag = item.tag.lower()

        if tag == 'nosecone':
            length = max(0.1, read_xml_float(item, 'length') * 1000.0)
            radius = read_xml_float(item, 'aftradius') * 1000.0
            if radius <= 0.1:
                radius = read_xml_float(item, 'outerradius') * 1000.0
            if radius <= 0.1:
                radius = read_xml_float(item, 'radius') * 1000.0
            radius = max(0.1, radius)
            shape = item.findtext('shape', 'ogive').lower()
            shape_param_elem = item.find('shapeparameter')
            shape_param = float(shape_param_elem.text) if (shape_param_elem is not None and shape_param_elem.text) else None

            shoulder_r = read_xml_float(item, 'aftshoulderradius') * 1000.0
            shoulder_l = read_xml_float(item, 'aftshoulderlength') * 1000.0

            component_list.append({
                'type': 'nosecone',
                'z_start': current_z,
                'length': length,
                'outer_radius': radius,
                'shape': shape,
                'shape_param': shape_param,
                'shoulder_radius': shoulder_r,
                'shoulder_length': shoulder_l
            })

            sub_container = item.find('subcomponents')
            if sub_container is not None:
                parse_ork_geometry(sub_container, current_z, component_list, parent_radius=radius, parent_length=length)
            current_z += length

        elif tag in ['bodytube', 'innertube']:
            length = max(0.1, read_xml_float(item, 'length') * 1000.0)
            radius = read_xml_float(item, 'outerradius') * 1000.0
            if radius <= 0.1:
                radius = read_xml_float(item, 'radius') * 1000.0
            if radius <= 0.1:
                radius = parent_radius
            radius = max(0.1, radius)

            thickness = read_xml_float(item, 'thickness') * 1000.0
            inner_r = read_xml_float(item, 'innerradius') * 1000.0
            if inner_r <= 0.0 and thickness > 0.0:
                inner_r = max(0.0, radius - thickness)
            if inner_r <= 0.0 and thickness <= 0.0:
                inner_r = max(0.0, radius - 1.5)
            if inner_r >= radius:
                inner_r = max(0.0, radius - 1.5)

            z_offset = get_component_z_offset(item, parent_length) if tag == 'innertube' else 0.0
            tube_z = current_z + z_offset

            component_list.append({
                'type': 'bodytube',
                'z_start': tube_z,
                'length': length,
                'outer_radius': radius,
                'inner_radius': inner_r
            })

            sub_container = item.find('subcomponents')
            if sub_container is not None:
                parse_ork_geometry(sub_container, tube_z, component_list, parent_radius=radius, parent_length=length)
            if tag == 'bodytube':
                current_z += length

        elif tag == 'transition':
            length = max(0.1, read_xml_float(item, 'length') * 1000.0)
            fore_r = max(0.1, read_xml_float(item, 'foreradius') * 1000.0)
            aft_r = max(0.1, read_xml_float(item, 'aftradius') * 1000.0)

            component_list.append({
                'type': 'transition',
                'z_start': current_z,
                'length': length,
                'fore_radius': fore_r,
                'aft_radius': aft_r
            })
            sub_container = item.find('subcomponents')
            if sub_container is not None:
                parse_ork_geometry(sub_container, current_z, component_list, parent_radius=aft_r, parent_length=length)
            current_z += length

        elif tag in ['trapezoidalfinset', 'trapezoidfinset', 'freeformfinset', 'ellipticalfinset', 'finset']:
            count = max(1, int(read_xml_float(item, 'fincount', 3)))
            z_offset = get_component_z_offset(item, parent_length)
            fin_z = current_z + z_offset
            thickness = max(0.8, read_xml_float(item, 'thickness') * 1000.0)
            radial_angle = read_xml_angle_degrees(item, 0.0)

            if tag == 'freeformfinset':
                custom_pts = parse_freeform_fin_points(item)
                if len(custom_pts) >= 3:
                    component_list.append({
                        'type': 'freeformfin',
                        'z_start': fin_z,
                        'count': count,
                        'pts': custom_pts,
                        'thickness': thickness,
                        'parent_radius': parent_radius,
                        'radial_angle': radial_angle
                    })
            elif tag == 'ellipticalfinset':
                rc = max(0.1, read_xml_float(item, 'rootchord') * 1000.0)
                height = max(0.1, read_xml_float(item, 'height') * 1000.0)
                component_list.append({
                    'type': 'ellipticalfin',
                    'z_start': fin_z,
                    'count': count,
                    'root_chord': rc,
                    'height': height,
                    'thickness': thickness,
                    'parent_radius': parent_radius,
                    'radial_angle': radial_angle
                })
            else:
                rc = max(0.1, read_xml_float(item, 'rootchord') * 1000.0)
                tc = max(0.0, read_xml_float(item, 'tipchord') * 1000.0)
                height = max(0.1, read_xml_float(item, 'height') * 1000.0)
                sweep = read_xml_float(item, 'sweep') * 1000.0
                if sweep == 0.0:
                    sweep_angle_elem = item.find('sweepangle')
                    if sweep_angle_elem is not None and sweep_angle_elem.text:
                        try:
                            sweep_deg = float(sweep_angle_elem.text)
                            sweep = height * math.tan(math.radians(sweep_deg))
                        except ValueError:
                            pass
                component_list.append({
                    'type': 'finset',
                    'z_start': fin_z,
                    'count': count,
                    'root_chord': rc,
                    'tip_chord': tc,
                    'height': height,
                    'sweep': sweep,
                    'thickness': thickness,
                    'parent_radius': parent_radius,
                    'radial_angle': radial_angle
                })

        elif tag in ['ring', 'centeringring', 'bulkhead', 'engineblock']:
            length = max(0.8, read_xml_float(item, 'length') * 1000.0)
            outer_r = read_xml_float(item, 'outerradius') * 1000.0
            if outer_r <= 0.1:
                outer_r = parent_radius - 0.2 if parent_radius > 0 else 10.0
            inner_r = read_xml_float(item, 'innerradius') * 1000.0 if tag not in ['bulkhead'] else 0.0

            z_offset = get_component_z_offset(item, parent_length)
            ring_z = current_z + z_offset
            component_list.append({
                'type': 'ring',
                'z_start': ring_z,
                'length': length,
                'outer_radius': outer_r,
                'inner_radius': inner_r
            })

        elif tag == 'launchlug':
            length = max(0.1, read_xml_float(item, 'length') * 1000.0)
            radius = read_xml_float(item, 'radius') * 1000.0
            if radius <= 0.05:
                radius = read_xml_float(item, 'outerradius') * 1000.0
            if radius <= 0.05:
                radius = 3.0
            angle = read_xml_angle_degrees(item, 0.0)
            z_offset = get_component_z_offset(item, parent_length)
            lug_z = current_z + z_offset
            component_list.append({
                'type': 'launchlug',
                'z_start': lug_z,
                'length': length,
                'radius': radius,
                'parent_radius': parent_radius,
                'angle': angle
            })

        elif tag == 'railbutton':
            outer_d = read_xml_float(item, 'outerdiameter') * 1000.0
            if outer_d <= 0.05:
                outer_d = read_xml_float(item, 'buttondiameter') * 1000.0
            outer_r = max(0.5, outer_d / 2.0 if outer_d > 0.05 else 4.0)
            total_h = read_xml_float(item, 'totalheight') * 1000.0
            if total_h <= 0.05:
                total_h = read_xml_float(item, 'height') * 1000.0
            if total_h <= 0.05:
                total_h = 6.0
            angle = read_xml_angle_degrees(item, 0.0)
            count = max(1, int(read_xml_float(item, 'instancecount', 1)))
            sep = read_xml_float(item, 'instanceseparation') * 1000.0
            z_offset = get_component_z_offset(item, parent_length)
            btn_z = current_z + z_offset
            component_list.append({
                'type': 'railbutton',
                'z_start': btn_z,
                'outer_radius': outer_r,
                'height': total_h,
                'parent_radius': parent_radius,
                'angle': angle,
                'count': count,
                'sep': sep
            })

        elif tag in ['masscomponent', 'shockcord']:
            name = item.findtext('name', tag.capitalize())
            length = read_xml_float(item, 'length') * 1000.0
            if length <= 0.05:
                length = read_xml_float(item, 'packedlength') * 1000.0
            if length <= 0.05:
                length = 10.0
            radius = read_xml_float(item, 'radius') * 1000.0
            if radius <= 0.05:
                radius = read_xml_float(item, 'packedradius') * 1000.0
            if radius <= 0.05:
                radius = parent_radius * 0.85 if parent_radius > 0 else 5.0
            z_offset = get_component_z_offset(item, parent_length)
            mc_z = current_z + z_offset
            component_list.append({
                'type': 'placeholder', 'label': name,
                'z_start': mc_z, 'length': length, 'radius': radius
            })

        elif tag in ['parachute', 'streamer']:
            name = item.findtext('name', tag.capitalize())
            length = read_xml_float(item, 'packedlength') * 1000.0
            if length <= 0.05:
                length = read_xml_float(item, 'length') * 1000.0
            if length <= 0.05:
                length = 20.0
            radius = read_xml_float(item, 'packedradius') * 1000.0
            if radius <= 0.05:
                radius = parent_radius * 0.85 if parent_radius > 0 else 5.0
            z_offset = get_component_z_offset(item, parent_length)
            p_z = current_z + z_offset
            component_list.append({
                'type': 'placeholder', 'label': name,
                'z_start': p_z, 'length': length, 'radius': radius
            })

        else:
            parse_ork_geometry(item, current_z, component_list, parent_radius=parent_radius, parent_length=parent_length)

    return component_list


def generate_step_cadquery(components, output_file="rocket.step"):
    """Genera el modelo CAD 3D sólido con CadQuery y exporta el archivo en formato STEP."""
    global CADQUERY_AVAILABLE
    if not CADQUERY_AVAILABLE:
        os.system("pip install cadquery")
        import cadquery as cq
        CADQUERY_AVAILABLE = True
    import cadquery as cq

    assembly = cq.Assembly()

    for idx, c in enumerate(components):
        try:
            c_type = c['type']
            z = c['z_start']

            if c_type == 'nosecone':
                r = c['outer_radius']
                l = c['length']
                shape_type = c.get('shape', 'ogive')
                shape_param = c.get('shape_param')

                clean = get_profile_points(shape_type, shape_param, r, l)
                solid = (cq.Workplane("XZ")
                         .polyline(clean)
                         .close()
                         .revolve(360)
                         .val()
                         .translate((0, 0, z)))

                assembly.add(solid, name=f"Ojiva_{idx}")

                sh_r = c.get('shoulder_radius', 0.0)
                sh_l = c.get('shoulder_length', 0.0)
                if sh_r > 0.1 and sh_l > 0.1:
                    shoulder = (cq.Workplane("XY")
                                .workplane(offset=z + l)
                                .circle(sh_r)
                                .extrude(sh_l))
                    assembly.add(shoulder, name=f"Ojiva_Hombro_{idx}")

            elif c_type == 'bodytube':
                r_out = c['outer_radius']
                r_in = c['inner_radius']
                l = c['length']
                if r_in > 0.1 and (r_out - r_in) > 0.1:
                    solid = cq.Workplane("XY").workplane(offset=z).circle(r_out).circle(r_in).extrude(l)
                else:
                    solid = cq.Workplane("XY").workplane(offset=z).circle(r_out).extrude(l)
                assembly.add(solid, name=f"Tubo_{idx}")

            elif c_type == 'transition':
                r_fore = c['fore_radius']
                r_aft = c['aft_radius']
                l = c['length']
                pts = remove_duplicate_points([(0.001, 0.0), (r_fore, 0.0), (r_aft, l), (0.001, l)])
                solid = (cq.Workplane("XZ")
                         .polyline(pts)
                         .close()
                         .revolve(360)
                         .val()
                         .translate((0, 0, z)))
                assembly.add(solid, name=f"Transicion_{idx}")

            elif c_type in ['finset', 'freeformfin', 'ellipticalfin']:
                count = c['count']
                th = c['thickness']
                p_rad = c.get('parent_radius', 0.0)
                base_angle = c.get('radial_angle', 0.0)

                if c_type == 'finset':
                    rc, tc, h, sw = c['root_chord'], c['tip_chord'], c['height'], c['sweep']
                    raw_pts = [
                        (p_rad, z),
                        (p_rad + h, z + sw),
                        (p_rad + h, z + sw + tc),
                        (p_rad, z + rc)
                    ]
                elif c_type == 'ellipticalfin':
                    rc, h = c['root_chord'], c['height']
                    raw_pts = [(p_rad, z)]
                    for i in range(1, 16):
                        theta = (math.pi / 15) * i
                        ri = p_rad + h * math.sin(theta)
                        zi = z + (rc / 2.0) * (1.0 - math.cos(theta))
                        raw_pts.append((ri, zi))
                else:
                    raw_pts = [(p_rad + pt[0], z + pt[1]) for pt in c['pts']]

                pts = remove_duplicate_points(raw_pts)
                if len(pts) >= 3:
                    base_fin = cq.Workplane("XZ").polyline(pts).close().extrude(th / 2.0, both=True)
                    for f in range(count):
                        angle = base_angle + f * (360.0 / count)
                        rot_fin = base_fin.rotate((0, 0, 0), (0, 0, 1), angle)
                        assembly.add(rot_fin, name=f"Aleta_{idx}_N{f+1}")

            elif c_type == 'ring':
                r_out = c['outer_radius']
                r_in = c['inner_radius']
                l = c['length']
                if r_in > 0.1 and (r_out - r_in) > 0.1:
                    solid = cq.Workplane("XY").workplane(offset=z).circle(r_out).circle(r_in).extrude(l)
                else:
                    solid = cq.Workplane("XY").workplane(offset=z).circle(r_out).extrude(l)
                assembly.add(solid, name=f"Anillo_{idx}")

            elif c_type == 'launchlug':
                r_lug = c['radius']
                l = c['length']
                p_rad = c['parent_radius']
                ang = c.get('angle', 0.0)
                solid = cq.Workplane("XY").workplane(offset=z).circle(r_lug).extrude(l)
                solid = solid.translate((p_rad + r_lug, 0, 0))
                solid = solid.rotate((0, 0, 0), (0, 0, 1), ang)
                assembly.add(solid, name=f"GuiaLanzamiento_{idx}")

            elif c_type == 'railbutton':
                r_btn = c['outer_radius']
                h = c['height']
                p_rad = c['parent_radius']
                ang = c.get('angle', 0.0)
                count = c.get('count', 1)
                sep = c.get('sep', 0.0)
                for n in range(count):
                    z_n = z + n * sep
                    btn = cq.Workplane("XY").circle(r_btn).extrude(h)
                    btn = btn.rotate((0, 0, 0), (0, 1, 0), 90)
                    btn = btn.translate((p_rad, 0, z_n))
                    btn = btn.rotate((0, 0, z_n), (0, 0, z_n + 1), ang)
                    assembly.add(btn, name=f"Boton_{idx}_{n+1}")

            elif c_type == 'placeholder':
                r = c['radius']
                l = c['length']
                label = c.get('label', 'Componente')
                solid = cq.Workplane("XY").workplane(offset=z).circle(r).extrude(l)
                assembly.add(solid, name=label)

        except Exception as err:
            print(f"Error en componente {idx} ({c.get('type')}): {err}")

    assembly.save(output_file)
    return output_file


def run_interactive_pipeline():
    """Ejecuta la interfaz interactiva de usuario en Colab para procesar y descargar el STEP."""
    print("=== Conversor de Archivos OpenRocket a CAD (STEP 3D Definitivo) ===")
    uploaded = colab_files.upload()
    if not uploaded:
        print("Operación cancelada.")
        return
    file_map = {name: data for name, data in uploaded.items() if name.endswith(('.ork', '.xml'))}
    if not file_map:
        print("No se encontraron archivos .ork o .xml válidos.")
        return

    file_dropdown = widgets.Dropdown(options=list(file_map.keys()), description='Archivo:')
    process_btn = widgets.Button(description='Generar STEP 3D', button_style='success')
    output_area = widgets.Output()

    def on_button_clicked(b):
        with output_area:
            clear_output()
            selected_file = file_dropdown.value
            print(f"Procesando geometría de: {selected_file} ...")
            raw_data = file_map[selected_file]
            xml_data = extract_xml_content(raw_data)
            root = ET.fromstring(xml_data)
            components = parse_ork_geometry(root)

            base_name = os.path.splitext(selected_file)[0]
            step_filename = f"{base_name}_3D.step"

            generate_step_cadquery(components, output_file=step_filename)
            print(f"¡Modelo STEP 3D generado con éxito!: {step_filename}")
            colab_files.download(step_filename)

    process_btn.on_click(on_button_clicked)
    display(widgets.VBox([file_dropdown, process_btn, output_area]))


run_interactive_pipeline()

=== Conversor de Archivos OpenRocket a CAD (STEP 3D Definitivo) ===


Saving ser.ork to ser (1).ork
